# Lesson: Foundations of Intelligent NetDevOps


### What you'll do

**1. Hands-on "AI essentials through a networking lens"**
- Your first prompt — parse IOS-XE BGP output with an LLM, no regex
- Same prompt, different OS — NX-OS, zero code changes

**2. Hands-on "From rules to learning"**
- Rules vs. learning on network counters — fixed threshold vs. Isolation Forest
- System prompts vs. user prompts — and why temperature pins at 0
- Few-shot prompting with network data — syslog classification by example
- From chain-of-thought to structured ACL output — reason first, then commit

### Requirements
- Python 3.10+
- A local [Ollama](https://ollama.com) server at `http://127.0.0.1:11434` with `gemma4:e4b` pulled (`ollama pull gemma4:e4b`)
- `scikit-learn` and `numpy` (see `requirements.txt`); no GPU required

### Running scenario
**IntentNet Corp** — ~600 IOS-XE switches, 3 campuses, BGP peering with 2 ISPs, 5 network engineers.

---

In [64]:
# ============================================================
# Environment Setup
# ============================================================
!pip install -q ollama scikit-learn numpy

import json
import ollama
from IPython.display import Markdown, display

# Configure Ollama client to point at the local server
OLLAMA_HOST = "http://127.0.0.1:11434"
MODEL = "gemma4:e4b"

client = ollama.Client(host=OLLAMA_HOST)


def show(response):
    """Render an Ollama chat response as Markdown in the notebook."""
    display(Markdown(response["message"]["content"]))


print(f"Environment ready. Using {MODEL} at {OLLAMA_HOST}")

213985.64s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Environment ready. Using gemma4:e4b at http://127.0.0.1:11434


---
## 1. Hands-on "AI essentials through a networking lens"

LLMs can parse unstructured network output that regex struggles with — let's prove it, in two parts:

1. **Your first prompt** — parse IOS-XE BGP output, no regex
2. **Same prompt, different vendor** — NX-OS, zero code changes

### Part 1 — Your First Prompt

Send a `show ip bgp summary` to the model and ask for structured data.

In [65]:
# ============================================================
# Sending a Basic Network Prompt
# ============================================================
# Sample show command output from IntentNet's edge router
bgp_output = """
BGP router identifier 10.1.1.1, local AS number 65001
BGP table version is 142, main routing table version 142
12 network entries using 2976 bytes of memory

Neighbor        V    AS MsgRcvd MsgSent   TblVer  InQ OutQ Up/Down  State/PfxRcd
192.168.1.1     4 65100   14523   14210      142    0    0 3d12h          8
192.168.2.1     4 65200    8891    8743      142    0    0 00:05:22       0
10.10.10.1      4 65001   22145   22098      142    0    0 7d03h          4
"""

response = client.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"Parse this BGP summary and list each neighbor "
                       f"with their AS, uptime, and status:\n\n{bgp_output}",
        }
    ],
    options={"temperature": 0},
)

show(response)

Based on the BGP summary provided, here is a list of each neighbor with their AS, uptime, and status:

| Neighbor IP | AS Number | Uptime | Status/Prefixes Received |
| :--- | :--- | :--- | :--- |
| **192.168.1.1** | 65100 | 3 days, 12 hours | Established (Received 8 prefixes) |
| **192.168.2.1** | 65200 | 5 minutes, 22 seconds | Unknown/Established (Received 0 prefixes) |
| **10.10.10.1** | 65001 | 7 days, 3 hours | Established (Received 4 prefixes) |

#### What just happened?

- **No regex** — the model parsed unstructured CLI directly
- It flagged the **anomaly** (0 prefixes, short uptime) unprompted — handy, but in a pipeline ask for it explicitly
- The same prompt works on NX-OS, IOS-XR, or Junos output

Compare that to maintaining a regex per OS and per release.

### Part 2 — Same Prompt, Different OS

The NX-OS format of the same data — different spacing, an `Active` state instead of a prefix count. Watch: **zero code changes**.

In [66]:
# ============================================================
# The Same Prompt, Different Vendor Output
# ============================================================
# NX-OS format of the same data — no code changes needed
nxos_bgp_output = """
BGP summary information for VRF default, address family IPv4 Unicast
BGP router identifier 10.1.1.1, local AS number 65001

Neighbor        V    AS    MsgRcvd    MsgSent   TblVer  InQ OutQ Up/Down  State/PfxRcd
192.168.1.1     4    65100    14523      14210      142    0    0 3d12h    8
192.168.2.1     4    65200     8891       8743      142    0    0 00:05:22 Active
"""

response = client.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"Parse this BGP summary and list each neighbor "
                       f"with their AS, uptime, and status:\n\n{nxos_bgp_output}",
        }
    ],
    options={"temperature": 0},
)

show(response)

Based on the BGP summary provided, here is the list of neighbors with their respective AS, uptime, and status:

| Neighbor IP | AS Number | Uptime/Duration | Status |
| :--- | :--- | :--- | :--- |
| **192.168.1.1** | 65100 | 3d12h (3 days, 12 hours) | Established (Up) |
| **192.168.2.1** | 65200 | 00:05:22 (5 minutes, 22 seconds) | Active |

Same analysis, correct handling of the NX-OS `Active` state — **vendor-agnostic and os-agnostic parsing, zero code changes**.

---
## 2. Hands-on "From rules to learning"

From deterministic rules to an AI-powered intent-to-ACL generator, in four parts:

1. **Rules vs. learning on network counters** — fixed threshold vs. Isolation Forest
2. **System prompts vs. user prompts** — and why temperature pins at 0
3. **Few-shot prompting with network data** — classification by example
4. **From chain-of-thought to structured ACL output** — reason first, then commit

### Part 1 — Rules vs. Learning on Network Counters

Same problem, two approaches — detecting unhealthy CRC counters:

- **Rule-based:** regex + fixed threshold. Catches what you anticipated; misses what you didn't.
- **Learning-based:** an anomaly detector trained on a historical baseline. Catches drift from "normal" even below your threshold.

LLMs are one kind of learning-based system — not the only one.

In [67]:
# ============================================================
# Rule-Based: Regex + Fixed Threshold on CRC Counters
# ============================================================
import re

# Hard-coded sample output from `show interface Gi0/1`
interface_output = """
GigabitEthernet0/1 is up, line protocol is up
  Hardware is iGbE, address is 0008.e3ff.fd01
  MTU 1500 bytes, BW 1000000 Kbit/sec
     12345678 packets input, 1234567890 bytes
     147 input errors, 142 CRC, 0 frame, 5 overrun, 0 ignored
"""

CRC_THRESHOLD = 100

match = re.search(r"(\d+) CRC", interface_output)
crc_count = int(match.group(1)) if match else 0

if crc_count > CRC_THRESHOLD:
    print(f"ALERT: {crc_count} CRC errors on Gi0/1 (threshold: {CRC_THRESHOLD})")
else:
    print(f"OK: {crc_count} CRC errors on Gi0/1 (under threshold of {CRC_THRESHOLD})")

ALERT: 142 CRC errors on Gi0/1 (threshold: 100)


The rule fires above 100 — but it has two failure modes:

1. **Below the threshold is invisible.** A jump from a baseline of 5 to 89 is suspicious; the rule says "ok."
2. **The threshold is a guess.** "Normal" on a core uplink isn't normal on an access port.

Now replace the fixed threshold with a model that **learns** what normal looks like.

In [68]:
# ============================================================
# Learning-Based: Isolation Forest on Counter History
# ============================================================
import numpy as np
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(42)

# 30 days of hourly CRC counts on Gi0/1 -- the historical baseline.
# Most readings are 0-14 with occasional blips up to 25.
baseline = rng.integers(0, 15, size=700)
blips = rng.integers(15, 26, size=20)
historical = np.concatenate([baseline, blips]).reshape(-1, 1)

model = IsolationForest(contamination=0.02, random_state=42).fit(historical)

# Today's hourly readings -- mix of normal and suspicious values
current = np.array([5, 12, 8, 89, 3, 142, 18, 0, 95, 10]).reshape(-1, 1)
predictions = model.predict(current)  # 1 = normal, -1 = anomaly

print(f"{'Reading':>8}  {'Rule-based':>12}  {'Learning-based':>15}")
print("-" * 41)
for value, pred in zip(current.flatten(), predictions):
    rule_verdict = "ALERT" if value > CRC_THRESHOLD else "ok"
    learn_verdict = "ALERT" if pred == -1 else "ok"
    print(f"{value:>8}  {rule_verdict:>12}  {learn_verdict:>15}")

 Reading    Rule-based   Learning-based
-----------------------------------------
       5            ok               ok
      12            ok               ok
       8            ok               ok
      89            ok            ALERT
       3            ok               ok
     142         ALERT            ALERT
      18            ok            ALERT
       0            ok               ok
      95            ok            ALERT
      10            ok               ok


**89** and **95** pass the rule (below 100) but Isolation Forest flags them — it learned that normal sits in the 0-25 range.

**18** is the flip side: inside the historical range, yet flagged — a **false positive**. Learning trades below-threshold blindness for occasional false alarms; that's why production systems keep rules as guardrails (the hybrid pattern).

#### When to use which

- **Rules:** well-defined logic, safety-critical actions, enumerable cases.
- **Learning:** pattern recognition, unstructured data, evolving baselines.
- **Hybrid:** rules as guardrails, AI inside those bounds.

> Rules define what you know. Learning discovers what you don't.

---
### Part 2 — System Prompts vs. User Prompts

- **System prompt:** role, context, constraints — the AI's **running-config**, persistent across interactions.
- **User prompt:** the specific task — an exec command.

In [69]:
# ============================================================
# Without a System Prompt
# ============================================================
# Ask for help with a network issue — no system prompt
response = client.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Interface GigabitEthernet0/1 is showing CRC errors. "
                       "What should I do?",
        }
    ],
    options={"temperature": 0},
)

display(Markdown("### Without System Prompt"))
show(response)

### Without System Prompt

CRC (Cyclic Redundancy Check) errors indicate that data packets received on the physical link have been corrupted during transmission. Essentially, the receiving device detects that the data stream is garbled or incomplete.

**Crucially, CRC errors are almost always a symptom of a physical layer problem, not a configuration error.** They mean the signal integrity is compromised somewhere between the two devices.

Here is a systematic, step-by-step troubleshooting plan, moving from the simplest checks to the most complex hardware replacements.

---

## 🛠️ Phase 1: Verification and Initial Checks (The Software Side)

Before touching any cables, gather data on the scope of the problem.

1.  **Check Error Rates:** Use your device's command line interface (CLI) to view the error counters for that specific port.
    *   *(Example Cisco Command: `show interfaces GigabitEthernet0/1`)*
2.  **Determine Scope:** Are the errors constant, or do they spike under load? Do other ports on the same switch show similar errors?
3.  **Check Link Status:** Verify that both ends of the link are reporting a stable "up" status and the expected speed (e.g., 1 Gbps).

## 🔌 Phase 2: Physical Layer Checks (The Most Common Culprits)

These steps address signal degradation, which is the primary cause of CRC errors.

### 1. Inspect Cables
*   **Damage:** Visually inspect the entire length of the cable for kinks, bends, or signs of physical damage.
*   **Length/Type:** Ensure the cable used is appropriate for the distance and media (e.g., using a Cat5e/Cat6 copper cable for 10 Gbps over too long a distance).
*   **Replacement:** **The single most effective step is to replace the patch cable.** Use a known good, certified cable of the correct type.

### 2. Check Connections and Cleanliness
*   **Seating:** Ensure the cable is fully seated in both ends (the switch port and the device/patch panel). Sometimes a slight misalignment can cause intermittent errors.
*   **Dust/Debris:** If using fiber optics, inspect the connectors for dust or dirt. Use appropriate cleaning tools to clean the end faces of the patch cables and transceivers.

### 3. Check Interference (EMI)
*   If the cable runs near power lines, motors, or other high-current electrical sources, it could be susceptible to Electromagnetic Interference (EMI). Try rerouting the cable away from these sources.

## ⚙️ Phase 3: Device and Configuration Checks (The Hardware Side)

If the physical link seems fine, check the settings on both ends of the connection.

### 1. Speed and Duplex Mismatch (CRITICAL CHECK)
*   **Problem:** If one side is manually set to `100 Mbps Full Duplex` and the other side is set to `Auto Negotiation`, they may negotiate incorrectly or fail to establish a stable link, leading to errors.
*   **Solution:** The best practice is almost always to leave both ends on **`Auto Negotiation`**. If you must manually configure them, ensure that *both* sides are configured identically (e.g., both set to `1 Gbps Full Duplex`).

### 2. Transceivers/SFPs (If using Fiber)
*   **Compatibility:** Ensure the SFP module installed in the switch port is compatible with the switch model and the fiber type being used.
*   **Failure:** A faulty or dirty transceiver can generate signal errors. **Try swapping the SFP module** with a known good unit to rule out hardware failure.

### 3. Port Failure Test
*   If you have another working port on the same switch, move the connection (and cable) to that new port. If the error persists, the issue is likely the connected device or the link itself.
*   If the error disappears when moved to a different port, the original port (`Gi0/1`) may be faulty and needs replacement.

## 🚨 Summary Checklist (Action Plan)

Follow these steps in order until the errors stop:

1.  ✅ **Check Configuration:** Verify both ends are set to `Auto Negotiation` for speed and duplex.
2.  ✅ **Replace Cable:** Swap out the patch cable connecting the two devices.
3.  ✅ **Clean Connectors:** Clean all fiber/copper connectors at both ends.
4.  ✅ **Swap Transceiver:** If applicable, swap the SFP module.
5.  ✅ **Move Port:** Move the connection to a different port on the switch.

If you have completed all these steps and the CRC errors persist, it is highly likely that either the **switch port itself** or the **device connected to the other end** has failed and requires replacement.

In [70]:
# ============================================================
# With a Network Engineer System Prompt
# ============================================================
# The four-line system prompt from the slides: Role, Context, Constraints, Output.
NETWORK_SYSTEM_PROMPT = (
    "You are a senior Cisco network engineer.\n"
    "Context: IntentNet Corp, 600 IOS-XE switches.\n"
    "Constraints: Only suggest IOS-XE commands.\n"
    "Output: Respond in structured JSON only."
)

response = client.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": NETWORK_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "Interface GigabitEthernet0/1 is showing CRC errors. "
                       "What should I do?",
        },
    ],
    options={"temperature": 0},
)

display(Markdown("### With Network Engineer System Prompt"))
show(response)

### With Network Engineer System Prompt

```json
{
  "issue": "CRC Errors on GigabitEthernet0/1",
  "severity": "High - Indicates physical layer integrity failure.",
  "recommendation_summary": "The primary cause of CRC errors is usually a faulty cable, dirty connector, or transceiver. The following steps are designed to diagnose the scope of the error and attempt to reset the link parameters via IOS-XE commands.",
  "troubleshooting_steps": [
    {
      "step": 1,
      "title": "Diagnosis and Data Collection (Verify Error Scope)",
      "description": "Run these commands first to confirm the rate of errors, check for associated input/output errors, and verify the physical status.",
      "commands": [
        "show interfaces GigabitEthernet0/1",
        "show interfaces counters GigabitEthernet0/1",
        "show logging | include CRC|input error"
      ],
      "action_note": "Analyze the output for high counts of 'CRC' or 'Input errors'. If these numbers are increasing rapidly, the physical link is unstable."
    },
    {
      "step": 2,
      "title": "Interface Reset (Soft Re-negotiation)",
      "description": "If the error rate is moderate and intermittent, cycling the interface can clear temporary errors and force a re-negotiation of speed/duplex.",
      "commands": [
        "configure terminal",
        "interface GigabitEthernet0/1",
        "shutdown",
        "no shutdown",
        "end"
      ],
      "action_note": "This is the least disruptive fix. If errors immediately return after this sequence, the issue is hardware or cabling related."
    },
    {
      "step": 3,
      "title": "Configuration Verification (If Errors Persist)",
      "description": "If the error persists after cycling, verify that speed and duplex settings are consistent across both ends. If possible, ensure 'auto' negotiation is used.",
      "commands": [
        "configure terminal",
        "interface GigabitEthernet0/1",
        "speed auto",
        "duplex auto",
        "end"
      ],
      "action_note": "Only use these commands if the link was previously configured with static speed/duplex settings. If 'auto' is already set, skip this step."
    },
    {
      "step": 4,
      "title": "Physical Layer Action (Non-Command)",
      "description": "If all IOS-XE commands fail to resolve the issue, the problem is almost certainly physical. The following actions must be performed manually:",
      "commands": [
        "N/A",
        "ACTION: Inspect and clean the fiber patch cable connectors.",
        "ACTION: Replace the transceiver (SFP/GBIC) on both ends of the link.",
        "ACTION: Replace the entire copper or fiber patch cable."
      ]
    }
  ]
}
```

#### Controlling Output: Temperature

Every call above pins `temperature: 0`. Temperature is the **wildcard mask for creativity**:

- `0.0.0.0` (temp 0) → deterministic, same output every run
- `0.0.255.255` (temp 1.0) → creative, varied

**Pin it at 0 for network automation** — a pipeline must give the same verdict for the same input. Save higher values for docs and brainstorming. The next cell proves it.

In [71]:
# ============================================================
# Temperature: 0.0 vs 1.0 on the Same Prompt
# ============================================================
# Same prompt twice at each temperature -- watch consistency change
prompt = "Suggest a hostname for a new core switch at the Madrid campus."

print("--- temperature = 0.0 (deterministic) ---")
for run in range(2):
    r = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    print(f"Run {run+1}:", r["message"]["content"].strip().splitlines()[0])

print("\n--- temperature = 1.0 (creative) ---")
for run in range(2):
    r = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 1.0},
    )
    print(f"Run {run+1}:", r["message"]["content"].strip().splitlines()[0])


--- temperature = 0.0 (deterministic) ---
Run 1: The best hostname depends heavily on your organization's existing network naming conventions (e.g., do you use hyphens, underscores, or periods? Do you prefer all caps?).
Run 2: The best hostname depends heavily on your organization's existing network naming conventions (e.g., do you use hyphens, underscores, or periods? Do you prefer all caps?).

--- temperature = 1.0 (creative) ---
Run 1: The best choice depends heavily on your existing organizational network naming conventions (SysAdmin culture vs. simple descriptive names).
Run 2: A good hostname should be **consistent, informative, brief, and easy to read.** It should tell anyone who sees it:


---
### Part 3 — Few-Shot Prompting with Network Data

Few-shot = show the model 2-3 input/output examples before the real query — like handing a junior engineer sample tickets. It locks the output format, which is critical for pipelines.

In [72]:
# ============================================================
# Few-Shot Syslog Classification
# ============================================================
# The two-field schema from the slides: severity + category.
few_shot_prompt = """Classify each syslog message into severity (critical/warning/info) and category (routing/interface/security/system).

Examples:
Input: "%OSPF-5-ADJCHG: Process 1, Nbr 10.1.1.2 on Gi0/1 from FULL to DOWN, Neighbor Down: Dead timer expired"
Output: {"severity": "critical", "category": "routing"}

Input: "%LINEPROTO-5-UPDOWN: Line protocol on Interface GigabitEthernet0/2, changed state to up"
Output: {"severity": "info", "category": "interface"}

Input: "%SEC-6-IPACCESSLOGP: list 101 denied tcp 10.5.3.22(44231) -> 172.16.1.1(22), 3 packets"
Output: {"severity": "warning", "category": "security"}

Now classify this message:
Input: "%BGP-3-NOTIFICATION: sent to neighbor 192.168.2.1 4/0 (hold time expired) 3 bytes"
Output:"""

response = client.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": NETWORK_SYSTEM_PROMPT},
        {"role": "user", "content": few_shot_prompt},
    ],
    options={"temperature": 0},
)

show(response)

```json
{
  "severity": "critical",
  "category": "routing"
}
```

The model replicated the exact pattern — **pipeline-compatible JSON** from free-text syslog.

Subtlety: the labels don't mirror the syslog digits (`OSPF-5` → **critical**, `SEC-6` → **warning**). The examples teach operational impact, not the number in the message.

---
### Part 4 — From Chain-of-Thought to Structured ACL Output

The core of this hands-on, in two steps: the model first **reasons** about a firewall intent, then **commits** to pipeline-ready JSON.

**Chain-of-Thought (CoT)** = ask the model to think step by step before answering — walking the troubleshooting tree out loud. It noticeably reduces errors on multi-step problems. Few-shot is *"here are example tickets"*; CoT is *"walk me through your reasoning before you close the ticket."*

In [73]:
# ============================================================
# Chain-of-Thought: Reasoning About an ACL Intent
# ============================================================
# NETWORK_SYSTEM_PROMPT demands JSON-only output, which would fight this
# hands-on step -- we want free-form reasoning, so we relax that constraint.
COT_SYSTEM_PROMPT = (
    "You are a senior Cisco network engineer.\n"
    "Context: IntentNet Corp, 600 IOS-XE switches.\n"
    "Constraints: Only suggest IOS-XE commands."
)

# Ask the LLM to walk through the intent step by step BEFORE producing output.
cot_prompt = """Analyze the following firewall intent step by step.
Do NOT output JSON yet -- just reason through each step.

1. Identify the action (permit or deny)
2. Extract source and destination networks
3. Determine the protocol and port
4. Note any ambiguity or assumption you had to make
5. State whether the intent can be expressed as a single ACL entry or needs multiple

Intent: "Block all SSH traffic from the guest network (10.0.0.0/8) to the
server network (172.16.0.0/16), except for the jump host at 10.0.0.50."
"""

response = client.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": COT_SYSTEM_PROMPT},
        {"role": "user", "content": cot_prompt},
    ],
    options={"temperature": 0},
)

show(response)

As a senior network engineer analyzing this firewall intent, I will break down the logic required to implement this policy using standard IOS-XE Access Control List (ACL) principles.

***

### Intent Analysis and Reasoning

**1. Identify the action (permit or deny)**
The primary goal is to **deny** SSH traffic generally. However, because of the explicit exception ("except for..."), the implementation requires a structured approach:
*   **Action 1:** Explicitly `permit` the allowed traffic flow (the jump host).
*   **Action 2:** Generally `deny` all other matching traffic flows (the block).

**2. Extract source and destination networks**
*   **General Source (Block Scope):** 10.0.0.0/8 (Guest Network)
*   **General Destination (Block Scope):** 172.16.0.0/16 (Server Network)
*   **Exception Source:** 10.0.0.50 (Jump Host)
*   **Exception Destination:** 172.16.0.0/16 (Server Network - implied, as the block applies to this range).

**3. Determine the protocol and port**
*   **Protocol:** TCP (Transmission Control Protocol).
*   **Port:** 22 (Standard SSH port).

**4. Note any ambiguity or assumption you had to make**
*   **Ambiguity/Scope:** The intent is highly specific to SSH traffic. I must assume that the ACL should *only* govern TCP port 22, and that other protocols (like ICMP or HTTP) are unaffected by this rule set.
*   **Assumption 1 (Crucial):** Since we are blocking all SSH from a large range but allowing one specific host, the policy structure must be: **Permit Specific $\rightarrow$ Deny General**. The order of rules in an ACL is critical and dictates the outcome.
*   **Assumption 2:** Standard SSH uses TCP port 22.

**5. State whether the intent can be expressed as a single ACL entry or needs multiple**
The intent **requires multiple entries** (at least three, including the implicit deny) to correctly implement the exception logic. A single rule cannot handle both the specific allowance and the general block simultaneously while maintaining proper security order.

***

### Summary of Logic Flow (Conceptual Implementation Order)

1.  **Rule 1 (Permit Exception):** Allow SSH from the jump host (10.0.0.50) to the server network (172.16.0.0/16).
2.  **Rule 2 (Deny Block):** Deny all other SSH traffic originating from the guest network (10.0.0.0/8) destined for the server network (172.16.0.0/16).
3.  **(Implicit Rule):** The final implicit `deny ip any any` handles everything else, ensuring that only explicitly permitted traffic passes through this ACL context.

#### Natural Language to Structured ACL Output

Now the commit step: natural-language firewall intent in, **JSON ACL rules** out — ready for Ansible or CI/CD. IntentNet's security team sends requests like *"Block all SSH traffic from the guest network"*; you build the translator.

In [74]:
# ============================================================
# Define the ACL Generation Prompt
# ============================================================
ACL_SYSTEM_PROMPT = """You are an ACL generator for Cisco IOS-XE devices at IntentNet Corp.

Network context:
- Guest network: 10.0.0.0/8
- Server network: 172.16.0.0/16
- Management network: 192.168.1.0/24
- Edge routers: BGP AS 65001
- Guest traffic enters on interface GigabitEthernet0/1
- Management traffic enters on interface GigabitEthernet0/2

When given a natural language firewall request, respond ONLY with valid JSON in this format:
{
  "acl_name": "string",
  "entries": [
    {
      "sequence": integer,
      "action": "permit" or "deny",
      "protocol": "tcp" or "udp" or "icmp" or "ip",
      "source": "CIDR notation",
      "destination": "CIDR notation or 'any'",
      "port": integer or null,
      "description": "string"
    }
  ],
  "apply_to": "the interface and direction to apply the ACL, e.g. GigabitEthernet0/1 inbound"
}

Do not include any explanation. Only output valid JSON."""


def generate_acl(intent: str) -> dict:
    """Convert natural language intent to structured ACL JSON."""
    response = client.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": ACL_SYSTEM_PROMPT},
            {"role": "user", "content": intent},
        ],
        options={"temperature": 0},
        format="json",
    )
    return json.loads(response["message"]["content"])


print("ACL generator function defined.")

ACL generator function defined.


In [75]:
# ============================================================
# Test 1: Simple Deny Rule
# ============================================================
intent_1 = "Block all SSH traffic from the guest network to the server network"
result_1 = generate_acl(intent_1)

print("Intent:", intent_1)
print("\nGenerated ACL:")
print(json.dumps(result_1, indent=2))

Intent: Block all SSH traffic from the guest network to the server network

Generated ACL:
{
  "acl_name": "GUEST_TO_SERVER_BLOCK",
  "entries": [
    {
      "sequence": 1,
      "action": "deny",
      "protocol": "tcp",
      "source": "10.0.0.0/8",
      "destination": "172.16.0.0/16",
      "port": 22,
      "description": "Block SSH from Guest to Server"
    }
  ],
  "apply_to": "GigabitEthernet0/1 inbound"
}


In [76]:
# ============================================================
# Test 2: Multiple Rules from a Single Request
# ============================================================
intent_2 = (
    "Allow HTTPS from the guest network to the server network, "
    "but block everything else from guest to management"
)
result_2 = generate_acl(intent_2)

print("Intent:", intent_2)
print("\nGenerated ACL:")
print(json.dumps(result_2, indent=2))

Intent: Allow HTTPS from the guest network to the server network, but block everything else from guest to management

Generated ACL:
{
  "acl_name": "GUEST_TO_SERVER_MGMT",
  "entries": [
    {
      "sequence": 10,
      "action": "permit",
      "protocol": "tcp",
      "source": "10.0.0.0/8",
      "destination": "172.16.0.0/16",
      "port": 443,
      "description": "Allow HTTPS from Guest to Server"
    },
    {
      "sequence": 20,
      "action": "deny",
      "protocol": "ip",
      "source": "10.0.0.0/8",
      "destination": "192.168.1.0/24",
      "port": null,
      "description": "Block all traffic from Guest to Management"
    }
  ],
  "apply_to": "GigabitEthernet0/1 inbound"
}


In [77]:
# ============================================================
# Verify Output is Pipeline-Ready
# ============================================================
# Demonstrate that the output can be used programmatically
acl = result_1

# Validate the schema before using the output -- a malformed field should
# fail loudly here, not silently downstream in the pipeline.
missing = {"acl_name", "entries", "apply_to"} - acl.keys()
assert not missing, f"Missing keys: {missing}"
for entry in acl["entries"]:
    assert entry["action"] in ("permit", "deny"), f"Invalid action: {entry}"
print("Schema validation passed.")
print()

print(f"ACL Name:        {acl.get('acl_name', 'N/A')}")
print(f"Number of entries: {len(acl.get('entries', []))}")
print(f"Apply to:        {acl.get('apply_to', 'N/A')}")
print()

for entry in acl.get("entries", []):
    action = entry.get("action", "")
    proto = entry.get("protocol", "")
    src = entry.get("source", "")
    dst = entry.get("destination", "")
    port = entry.get("port", "")
    port_str = f" eq {port}" if port else ""
    print(f"  {entry.get('sequence', '')} {action} {proto} {src} {dst}{port_str}")
    print(f"     ! {entry.get('description', '')}")

print()
print("This JSON can be fed directly into an Ansible playbook or CI/CD validation step.")

Schema validation passed.

ACL Name:        GUEST_TO_SERVER_BLOCK
Number of entries: 1
Apply to:        GigabitEthernet0/1 inbound

  1 deny tcp 10.0.0.0/8 172.16.0.0/16 eq 22
     ! Block SSH from Guest to Server

This JSON can be fed directly into an Ansible playbook or CI/CD validation step.


---
## Lesson Summary

You:

1. **Parsed BGP output** with an LLM — no regex, two vendors
2. **Contrasted rules vs. learning** — fixed threshold vs. Isolation Forest on CRC counters
3. **Anchored behavior with a system prompt** — and pinned temperature at 0
4. **Classified syslog with few-shot prompting** — consistent structured output
5. **Reasoned with chain-of-thought, then built an ACL generator** — pipeline-ready JSON from plain English; the logic is a ~12-line function, the intelligence lives in the system prompt

**Key insight:** the LLM doesn't replace your network expertise — it amplifies it. You still need to know what a valid ACL looks like to verify the output.